In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
from model.vectorizer_topic_clusterizer import TopicVectorizerClusterizer

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from scipy.spatial.distance import pdist
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

pio.templates.default = "plotly_white"

In [ ]:
tvc = TopicVectorizerClusterizer(11, 0.0002613762477788285, k_neighbours_inference=1)
train_data = pd.read_csv(r'C:\Users\aleks\OneDrive\Desktop\Studying\your-book-finder\data\raw\gutenberq_books_tiny.csv')
result = tvc.fit_transform(train_data)

print(f"Silhouette score: {silhouette_score(np.stack(result['vector'].values), result['cluster'].values)}")

In [ ]:
tsne = TSNE(n_components=2, learning_rate='auto', init='pca', perplexity=15, max_iter=1000)

In [ ]:
vector = tvc.predict("The Act of Declaration of Independence of Ukraine",
                "The Act of Declaration of Independence of Ukraine was adopted by the Supreme Soviet of the Ukrainian "
                "SSR (Verkhovna Rada) on August 24, 1991. This act declared Ukraine's independence from the Soviet "
                "Union. It was later affirmed by a national referendum on December 1, 1991, where a majority of "
                "Ukrainians across all regions supported the declaration.")

result = pd.concat([result, vector], axis=0)

In [ ]:
vectors_embedded = pd.DataFrame(tsne.fit_transform(np.stack(result['vector'].values)), 
                                index=result['vector'].index)

In [ ]:
fig = go.Figure()
clusters = result['cluster'].unique()

for cluster in clusters:
    titles = list(result[result['cluster'] == cluster].index)
    data = vectors_embedded.loc[titles, :]
    
    fig.add_trace(go.Scatter(x=data.iloc[:, 0], y=data.iloc[:, 1], 
                             mode='markers', 
                             marker=dict(size=10, 
                                         color=cluster),
                             text=titles,
                             hovertemplate='Title: %{text}<extra></extra>',
                             hoverinfo='text'))

fig.update_layout(margin=dict(l=25, r=25, t=35, b=35), 
                  plot_bgcolor='white',
                  paper_bgcolor='white',
                  xaxis=dict(showgrid=False, showticklabels=False, zeroline=False),
                  yaxis=dict(showgrid=False, showticklabels=False, zeroline=False))
fig.show()